# 06 — Open Source Models — Practical Examples

**Covers**: Ollama local setup, OpenAI-compatible OSS APIs, Groq speed benchmark, model comparison, cost comparison OSS vs proprietary

In [ ]:
import os, json, time
from openai import OpenAI
from dotenv import load_dotenv
from rich import print as rprint

load_dotenv()
client = OpenAI()  # Proprietary (OpenAI)
MODEL = 'gpt-4o-mini'
print('✅ Setup complete')

## 1. Ollama — Local Model (OpenAI-Compatible API)

In [ ]:
# Prerequisite: Install Ollama from https://ollama.ai
# Then: ollama pull llama3.1:8b

try:
    ollama_client = OpenAI(
        base_url='http://localhost:11434/v1',
        api_key='ollama'  # Required but ignored by Ollama
    )
    PROMPT = 'Explain Python generators in 3 sentences.'
    
    t0 = time.perf_counter()
    r = ollama_client.chat.completions.create(
        model='llama3.1:8b',  # Must be pulled first: ollama pull llama3.1:8b
        messages=[
            {'role': 'system', 'content': 'You are a helpful Python teacher.'},
            {'role': 'user',   'content': PROMPT}
        ],
        max_tokens=150
    )
    elapsed = (time.perf_counter() - t0) * 1000
    print('=== Ollama (llama3.1:8b — local) ===')
    print(f'Latency: {elapsed:.0f}ms (depends on hardware)')
    print(f'Response: {r.choices[0].message.content[:250]}')
    print('💡 100% local — no data leaves your machine, no API cost')
except Exception as e:
    print('📌 To use Ollama:')
    print('   1. Install: https://ollama.ai')
    print('   2. Pull model: ollama pull llama3.1:8b')
    print('   3. Ollama serves OpenAI-compatible API at localhost:11434')
    print(f'   Error: {type(e).__name__}: {e}')

## 2. Groq — Fastest OSS Inference

In [ ]:
try:
    from groq import Groq
    groq_client = Groq(api_key=os.getenv('GROQ_API_KEY', ''))
    PROMPT = 'Explain Python generators in 3 sentences.'
    
    t0 = time.perf_counter()
    r = groq_client.chat.completions.create(
        model='llama-3.1-70b-versatile',
        messages=[
            {'role': 'system', 'content': 'You are a helpful Python teacher.'},
            {'role': 'user',   'content': PROMPT}
        ],
        max_tokens=150
    )
    elapsed = (time.perf_counter() - t0) * 1000
    out_tokens = r.usage.completion_tokens
    tps = out_tokens / (r.usage.completion_time if hasattr(r.usage, 'completion_time') else elapsed/1000)
    
    print('=== Groq (llama-3.1-70b) ===')
    print(f'Total latency:  {elapsed:.0f}ms')
    print(f'Output tokens:  {out_tokens}')
    print(f'Response: {r.choices[0].message.content[:250]}')
except ImportError:
    print('📌 pip install groq')
    print('   Sign up at console.groq.com for a free API key')
    print('   Groq runs Llama 3.1 at 300-750 tokens/sec (fastest available)')
except Exception as e:
    print(f'Check GROQ_API_KEY: {type(e).__name__}')

## 3. OSS vs Proprietary — Side-by-Side Quality Check

In [ ]:
# Compare quality: gpt-4o-mini (proprietary) vs simulated OSS response
# (In real setup, swap proprietary_client/oss_client below)

BENCH_TASKS = [
    {
        'id': 'Q1',
        'prompt': 'Write a Python function to flatten a nested list of arbitrary depth.',
        'check': lambda r: 'def ' in r and 'return' in r
    },
    {
        'id': 'Q2',
        'prompt': 'In one sentence: what is the difference between a process and a thread?',
        'check': lambda r: len(r.split()) > 5
    },
    {
        'id': 'Q3',
        'prompt': 'Classify as POSITIVE, NEGATIVE, or NEUTRAL: "The Python documentation could be clearer for beginners."',
        'check': lambda r: any(w in r.upper() for w in ['POSITIVE', 'NEGATIVE', 'NEUTRAL'])
    },
]

results = {}
for task in BENCH_TASKS:
    t0 = time.perf_counter()
    r = client.chat.completions.create(
        model=MODEL,
        messages=[{'role': 'user', 'content': task['prompt']}],
        max_tokens=150, temperature=0
    )
    elapsed = (time.perf_counter() - t0) * 1000
    content = r.choices[0].message.content
    passed = task['check'](content)
    results[task['id']] = {'passed': passed, 'ms': elapsed}
    print(f"[{task['id']}] {'✅' if passed else '❌'} ({elapsed:.0f}ms): {content[:80]}...")

## 4. Cost Comparison — OSS (API-hosted) vs Proprietary

In [ ]:
# OSS models via cloud APIs vs proprietary at scale
PRICING = {
    # Proprietary
    'gpt-4o':            {'input': 5.00e-6,   'output': 15.00e-6,  'type': 'proprietary'},
    'gpt-4o-mini':       {'input': 0.15e-6,   'output': 0.60e-6,   'type': 'proprietary'},
    'claude-3-5-sonnet': {'input': 3.00e-6,   'output': 15.00e-6,  'type': 'proprietary'},
    # OSS via cloud
    'llama-3.1-70b (Groq)':  {'input': 0.59e-6,'output': 0.79e-6,  'type': 'oss-cloud'},
    'llama-3.1-8b (Groq)':   {'input': 0.05e-6,'output': 0.08e-6,  'type': 'oss-cloud'},
    'deepseek-v3 (API)':     {'input': 0.27e-6,'output': 1.10e-6,  'type': 'oss-cloud'},
    # OSS self-hosted (server amortized)
    'llama-3.1-70b (local)': {'input': 0.02e-6,'output': 0.02e-6,  'type': 'oss-local'},
}

IN_T, OUT_T, N = 2000, 400, 1_000_000  # 1M calls
print(f"Cost for {N:,} calls × ({IN_T} input + {OUT_T} output tokens)")
print(f"{'Model':<30} {'Type':<12} {'Per call':>10} {'1M calls':>12}")
print('-' * 68)
for m, p in PRICING.items():
    per_call = IN_T * p['input'] + OUT_T * p['output']
    total = per_call * N
    icon = '🔒' if p['type']=='proprietary' else ('🌐' if 'cloud' in p['type'] else '🖥️')
    print(f"{icon} {m:<28} {p['type']:<12} ${per_call:>9.5f} ${total:>11,.0f}")

## 5. When to Go Self-Hosted — Break-Even Analysis

In [ ]:
# GPU server cost vs API cost
# A100 80GB GPU server: ~$3.00/hr (cloud rental)
# Can serve llama-3.1-70b at ~50 tokens/sec

GPU_COST_PER_HR     = 3.00     # $/hr for A100 80GB
GPU_TOKENS_PER_SEC  = 50       # throughput for 70B model
GPU_TOKENS_PER_HR   = GPU_TOKENS_PER_SEC * 3600  # = 180,000
COST_PER_TOKEN_GPU  = GPU_COST_PER_HR / GPU_TOKENS_PER_HR

# gpt-4o-mini cost per output token
COST_PER_TOKEN_OAI = 0.60e-6

print('Self-hosted vs gpt-4o-mini break-even analysis')
print('─' * 55)
print(f'Self-hosted (A100 80GB):')
print(f'  GPU cost:      ${GPU_COST_PER_HR}/hr')
print(f'  Throughput:    {GPU_TOKENS_PER_SEC} tokens/sec = {GPU_TOKENS_PER_HR:,} tokens/hr')
print(f'  Cost/token:    ${COST_PER_TOKEN_GPU:.8f}')
print(f'\ngpt-4o-mini (cloud API):')
print(f'  Cost/token:    ${COST_PER_TOKEN_OAI:.8f}')
print()

if COST_PER_TOKEN_GPU < COST_PER_TOKEN_OAI:
    ratio = COST_PER_TOKEN_OAI / COST_PER_TOKEN_GPU
    breakeven_tokens_day = GPU_COST_PER_HR * 24 / COST_PER_TOKEN_OAI
    print(f'✅ Self-hosted is {ratio:.0f}× cheaper per token when GPU is running')
    print(f'💡 Break-even: need {breakeven_tokens_day:,.0f} tokens/day to justify 24/7 server')
else:
    print(f'📌 Cloud API is cheaper at this scale')

## 📌 Summary

- **Ollama** = easiest local setup; `base_url='http://localhost:11434/v1'` gives OpenAI-compatible API
- **Groq** = fastest cloud OSS inference (Llama, Mixtral); same `.completion()` API as OpenAI
- **Together AI, Replicate** = easy cloud hosting of any open model
- **OSS quality** = Llama 3.1 70B ≈ GPT-4-turbo quality at 10× lower cost on cloud
- **Self-hosting breaks even** at ~1M tokens/day — below that, cloud APIs win on cost
- **Privacy** = the strongest reason to self-host regardless of cost